Colab 1 — Full fine-tuning (FFT) with a tiny model

In [ ]:
# %% [markdown]
# Part 1 — Full Finetuning (SFT) with SmolLM2-135M (Unsloth + TRL)
# - Full finetune (no LoRA)
# - FP32 training (no AMP)
# - Clean chat template (no repeated user turns)
# - report_to="none" (no external logging)
# - End-to-end: install → data → train → save → inference

# %%capture
%pip -q install -U unsloth transformers accelerate datasets peft bitsandbytes trl

# --- Hard-disable any mixed precision via Accelerate ---
import os
os.environ["ACCELERATE_MIXED_PRECISION"] = "no"
os.environ["ACCELERATE_USE_DEEPSPEED"] = "false"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# --- Hygiene & seeds ---
import random, numpy as np, torch
def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
set_seed(42)

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    # optional: small FP32 speed bump
    torch.backends.cuda.matmul.allow_tf32 = True

# === 1) Tiny demo dataset (replace with your own) ===
from datasets import Dataset

rows = [
    {"messages": [
        {"role":"system","content":"You are a concise, helpful coding assistant."},
        {"role":"user","content":"Write a Python function to reverse a string."},
        {"role":"assistant","content":"def reverse_str(s):\n    return s[::-1]"},
    ]},
    {"messages": [
        {"role":"system","content":"You are a helpful assistant."},
        {"role":"user","content":"Explain what a lambda function is in Python."},
        {"role":"assistant","content":"A lambda is an anonymous inline function defined with the `lambda` keyword, e.g., lambda x: x + 1."},
    ]},
    {"messages": [
        {"role":"system","content":"You are a helpful assistant."},
        {"role":"user","content":"Give me a short example of list comprehension."},
        {"role":"assistant","content":"squares = [x*x for x in range(5)]  # [0, 1, 4, 9, 16]"},
    ]},
]
raw_ds = Dataset.from_list(rows)

# === 2) Load model/tokenizer with Unsloth (no 4-bit; we want full FT in FP32) ===
from unsloth import FastLanguageModel

BASE_MODEL = "HuggingFaceTB/SmolLM2-135M"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = BASE_MODEL,
    max_seq_length = 2048,
    load_in_4bit   = False,   # full FT in FP32
)

# Clean, robust chat template (no duplication when add_generation_prompt=True)
if getattr(tokenizer, "chat_template", None) in (None, ""):
    tokenizer.chat_template = r"""{%- for message in messages -%}
{%- if message['role'] == 'system' -%}
System: {{ message['content'] }}
{%- elif message['role'] == 'user' -%}
User: {{ message['content'] }}
{%- elif message['role'] == 'assistant' -%}
Assistant: {{ message['content'] }}
{%- endif -%}
{%- endfor -%}
{%- if add_generation_prompt -%}
Assistant:
{%- endif -%}"""

# Ensure a pad token exists
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token or "<|pad|>"
    try: model.config.pad_token_id = tokenizer.pad_token_id
    except: pass
tokenizer.padding_side = "right"

# Convert messages → training text
def to_text(example):
    return tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,   # include assistant targets
    )
train_ds = raw_ds.map(lambda e: {"text": to_text(e)})
print("Preview first sample:\n", train_ds[0]["text"][:300])

# Force model params to FP32 and trainable (full FT)
model.to(dtype=torch.float32)
for p in model.parameters():
    p.requires_grad = True
    if p.data.dtype != torch.float32:
        p.data = p.data.to(torch.float32)

# (optional) VRAM saver
if hasattr(model, "gradient_checkpointing_enable"):
    model.gradient_checkpointing_enable()

# === 3) Train with TRL SFTTrainer (pure FP32; no AMP) ===
from transformers import TrainingArguments
from trl import SFTTrainer

args = TrainingArguments(
    output_dir="ft-smollm2-135m-full",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=1,            # increase for real training
    learning_rate=2e-4,
    logging_steps=10,
    save_steps=100,
    fp16=False, bf16=False,        # <- keep mixed precision OFF
    report_to="none",              # <- no W&B or others
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    dataset_text_field="text",
    max_seq_length=1024,
    args=args,
)

trainer.train()

# Save artifacts
out_dir = "ft-smollm2-135m-full"
trainer.save_model(out_dir)
tokenizer.save_pretrained(out_dir)
print(f"Saved to: {out_dir}")

# === 4) Quick inference (prints only the assistant's response) ===
def chat(prompt, system="You are a helpful assistant.", max_new_tokens=128, temperature=0.7, top_p=0.9):
    messages = [
        {"role":"system","content":system},
        {"role":"user","content":prompt},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=top_p,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
        )
    decoded = tokenizer.decode(out[0], skip_special_tokens=True)
    # Return only what the assistant wrote after the final "Assistant:"
    print(decoded.split("Assistant:")[-1].strip())

# --- remove the WANDB deprecation warning if your env sets it elsewhere
import os; os.environ.pop("WANDB_DISABLED", None)

# Robust inference: clear stop rules + anti-repetition + clean extraction
from transformers import StoppingCriteria, StoppingCriteriaList

def make_stop_criteria(prompt_ids, stop_strings):
    stop_ids = [tokenizer.encode(s, add_special_tokens=False) for s in stop_strings]
    class StopOnStrings(StoppingCriteria):
        def __init__(self, start_len): self.start_len = start_len
        def __call__(self, input_ids, scores, **kwargs):
            gen = input_ids[0][self.start_len:]
            text = tokenizer.decode(gen, skip_special_tokens=True)
            return any(s in text for s in stop_strings)
    return StoppingCriteriaList([StopOnStrings(start_len=prompt_ids.shape[1])])

def extract_assistant(text: str) -> str:
    # take what comes after the final "Assistant:" and trim anything after a new "User:"/"System:"
    out = text.split("Assistant:")[-1]
    for cut in ("\nUser:", "\nSystem:"):
        if cut in out:
            out = out.split(cut)[0]
    return out.strip()

def chat(prompt, system="You are a helpful assistant.",
         max_new_tokens=128, temperature=0.7, top_p=0.9):
    messages = [
        {"role":"system","content":system},
        {"role":"user","content":prompt},
    ]
    prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)

    stop_strings = ["\nUser:", "\nSystem:"]
    stopping = make_stop_criteria(inputs["input_ids"], stop_strings)

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=top_p,
            repetition_penalty=1.1,      # curb loops
            no_repeat_ngram_size=3,      # curb short repeats
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
            stopping_criteria=stopping,
        )
    decoded = tokenizer.decode(out[0], skip_special_tokens=True)
    print(extract_assistant(decoded))

    # --- remove the WANDB deprecation warning if your env sets it elsewhere
import os; os.environ.pop("WANDB_DISABLED", None)

# Robust inference: clear stop rules + anti-repetition + clean extraction
from transformers import StoppingCriteria, StoppingCriteriaList

def make_stop_criteria(prompt_ids, stop_strings):
    stop_ids = [tokenizer.encode(s, add_special_tokens=False) for s in stop_strings]
    class StopOnStrings(StoppingCriteria):
        def __init__(self, start_len): self.start_len = start_len
        def __call__(self, input_ids, scores, **kwargs):
            gen = input_ids[0][self.start_len:]
            text = tokenizer.decode(gen, skip_special_tokens=True)
            return any(s in text for s in stop_strings)
    return StoppingCriteriaList([StopOnStrings(start_len=prompt_ids.shape[1])])

def extract_assistant(text: str) -> str:
    # take what comes after the final "Assistant:" and trim anything after a new "User:"/"System:"
    out = text.split("Assistant:")[-1]
    for cut in ("\nUser:", "\nSystem:"):
        if cut in out:
            out = out.split(cut)[0]
    return out.strip()

def chat(prompt, system="You are a helpful assistant.",
         max_new_tokens=128, temperature=0.7, top_p=0.9):
    messages = [
        {"role":"system","content":system},
        {"role":"user","content":prompt},
    ]
    prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)

    stop_strings = ["\nUser:", "\nSystem:"]
    stopping = make_stop_criteria(inputs["input_ids"], stop_strings)

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=top_p,
            repetition_penalty=1.1,      # curb loops
            no_repeat_ngram_size=3,      # curb short repeats
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
            stopping_criteria=stopping,
        )
    decoded = tokenizer.decode(out[0], skip_special_tokens=True)
    print(extract_assistant(decoded))




print("\n=== Sample 1 ===")
chat("Write a Python function to reverse a string.")

print("\n=== Sample 2 ===")
chat("Explain what a lambda function is in Python.")




CUDA available: True
GPU: Tesla T4


/tmp/ipython-input-619925631.py:54: UserWarning: WARNING: Unsloth should be imported before transformers to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.11.2: Fast Llama patching. Transformers: 4.57.1.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
HuggingFaceTB/SmolLM2-135M does not have a padding token! Will use pad_token = <|endoftext|>.


Map:   0%|          | 0/3 [00:00<?, ? examples/s]

num_proc must be <= 3. Reducing num_proc to 3 for dataset of size 3.


Preview first sample:
 System: You are a concise, helpful coding assistant.User: Write a Python function to reverse a string.Assistant: def reverse_str(s):
    return s[::-1]


Unsloth: Tokenizing ["text"] (num_proc=3):   0%|          | 0/3 [00:00<?, ? examples/s]

The model is already on multiple devices. Skipping the move to device specified in `args`.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 3 | Num Epochs = 1 | Total steps = 1
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 134,515,008 of 134,515,008 (100.00% trained)


Step,Training Loss


Saved to: ft-smollm2-135m-full

=== Sample 1 ===
This is the user's last name, which they will use for their email address in the next step.User Input: The first line of input contains three numbers separated by spaces: two integers A and B (A > 0).The second line of output consists of a single integer N, representing the number of characters that need to be removed from the original string.The third line of printout consists of the character before all other characters as well as one more space, where N < 20.You should put these values into the variable strInputList.For example, if you enter "C" then you get the following

=== Sample 2 ===
User: explain what a list is in C#?
50.17632948 26.02672299 100.00% 23.51 %

50 Related Questions

What is a lambda expression?

A lambda expression is used to create an anonymous function that takes no arguments and returns only the result of its body. In other words, it’s just like a function but with no parameters or return value. It can be very 